In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lianapestereva/input-dataset/test_inputs.jsonl
/kaggle/input/datasets/lianapestereva/input-dataset/train_type1.jsonl
/kaggle/input/datasets/lianapestereva/input-dataset/train_type2.jsonl
/kaggle/input/datasets/lianapestereva/input-dataset-2/test_inputs.jsonl
/kaggle/input/datasets/lianapestereva/input-dataset-2/train_type1.jsonl
/kaggle/input/datasets/lianapestereva/input-dataset-2/train_type2.jsonl


In [37]:
import os
os.environ["UNSLOTH_USE_FUSED_LOSS"] = "0"

!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install -q --no-deps xformers
!pip install -q peft accelerate bitsandbytes datasets trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [38]:
import os
import json
import torch
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/lianapestereva/input-dataset-2") 
TRAIN_TYPE1_PATH = DATA_DIR / "train_type1.jsonl"
TRAIN_TYPE2_PATH = DATA_DIR / "train_type2.jsonl"
TEST_INPUTS_PATH = DATA_DIR / "test_inputs.jsonl"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPTS = {
    "type1": (
        "Ты — редактор Telegram-канала @banki_oil. "
        "Перепиши новостной текст в стиле этого канала: короткий ироничный аналитический пост, "
        "без воды, с акцентом на суть и с фирменной ироничной концовкой. "
        "В конце добавь @banki_oil."
    ),
    "type2": (
        "Ты — редактор городского Telegram-канала @moscowach. "
        "Перепиши новостной текст в городском новостном формате: эмодзи в начале, "
        "жирный короткий заголовок, затем 1–2 коротких абзаца по существу. "
        "Пиши живо и по-городскому, без лишней аналитики."
    ),
}

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  
MAX_SEQ_LENGTH = 4096

print("Data dir exists:", DATA_DIR.exists())
print("Train type1 exists:", TRAIN_TYPE1_PATH.exists())
print("Train type2 exists:", TRAIN_TYPE2_PATH.exists())
print("Test inputs exists:", TEST_INPUTS_PATH.exists())

Data dir exists: True
Train type1 exists: True
Train type2 exists: True
Test inputs exists: True


In [39]:

def load_any_json(path: Path):
    """
    Читает файл и возвращает список объектов.
    Поддерживает:
      - JSON-массив:   [ {...}, {...} ]
      - одиночный объект: { ... }
      - JSON Lines:    {...}\n{...}\n{...}
    """
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()

    if not content:
        return []

    try:
        data = json.loads(content)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return [data]
    except json.JSONDecodeError:
        pass

    items = []
    for lineno, line in enumerate(content.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        try:
            items.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Не удалось распарсить строку {lineno} в {path}: {e}\n{line[:200]}")
    return items


def load_train_examples(path: Path, style: str):
    raw = load_any_json(path)
    examples = []
    for i, item in enumerate(raw):
        if not isinstance(item, dict) or "input" not in item or "output" not in item:
            raise ValueError(f"Элемент #{i} в {path} не содержит input/output: {item}")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPTS[style]},
            {"role": "user", "content": item["input"]},
            {"role": "assistant", "content": item["output"]},
        ]
        examples.append({"messages": messages})
    return examples


def load_test_inputs(path: Path):
    raw = load_any_json(path)
    texts = []
    for item in raw:
        if isinstance(item, dict):
            text = item.get("input") or item.get("text") or item.get("news")
            if text is None:
                text = " ".join(str(v) for v in item.values())
            texts.append(text)
        else:
            texts.append(str(item))
    return texts


train_data_type1 = load_train_examples(TRAIN_TYPE1_PATH, "type1")
train_data_type2 = load_train_examples(TRAIN_TYPE2_PATH, "type2")
test_texts = load_test_inputs(TEST_INPUTS_PATH)

print(f"Type1 examples: {len(train_data_type1)}")
print(f"Type2 examples: {len(train_data_type2)}")
print(f"Test inputs: {len(test_texts)}")
print("\n--- Пример messages (type1) ---")
print(json.dumps(train_data_type1[0]["messages"], ensure_ascii=False, indent=2)[:800])

Type1 examples: 100
Type2 examples: 100
Test inputs: 5

--- Пример messages (type1) ---
[
  {
    "role": "system",
    "content": "Ты — редактор Telegram-канала @banki_oil. Перепиши новостной текст в стиле этого канала: короткий ироничный аналитический пост, без воды, с акцентом на суть и с фирменной ироничной концовкой. В конце добавь @banki_oil."
  },
  {
    "role": "user",
    "content": "Эксперт Комитета Государственной думы по развитию гражданского общества, вопросам общественных и религиозных объединений Игорь Иванишко выступил с резкой критикой восточной поп-культуры. По его словам, сатанинские и деструктивные культы целенаправленно используют мангу и аниме для манипуляции сознанием несовершеннолетних граждан РФ. Эксперт подчеркнул, что яркие визуальные образы привлекают детскую аудиторию, после чего происходит скрытое воздействие на неокрепшую психику. В связи с эти


In [40]:

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,        
    dtype=None,               

if tokenizer.chat_template is None:
    raise ValueError(
        "У токенизатора нет chat_template. "
        "Убедитесь, что используете Instruct-версию модели."
    )

def make_lora_model(base_model):
    return FastLanguageModel.get_peft_model(
        base_model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

model = make_lora_model(model)
print("LoRA model is ready.")

==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
LoRA model is ready.


In [41]:

from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import gc

def train_style(style_key: str, train_examples: list, output_adapter_dir: Path):
    """
    Обучает LoRA-адаптер для одного стиля.
    style_key: "type1" или "type2"
    train_examples: список словарей {"messages": [...]}
    output_adapter_dir: куда сохранить адаптер
    """
    global model, tokenizer

    print(f"\n{'='*60}")
    print(f"Training style: {style_key}")
    print(f"{'='*60}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )
    model = make_lora_model(model)

    dataset = Dataset.from_list(train_examples)

    sft_config = SFTConfig(
        output_dir=str(output_adapter_dir / "checkpoints"),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=6,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field=None,   # будем использовать messages
        packing=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=sft_config,
    )

    trainer.train()

    output_adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(output_adapter_dir))
    tokenizer.save_pretrained(str(output_adapter_dir))

    print(f"Adapter saved to: {output_adapter_dir}")
    return model, tokenizer

In [42]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA_VISIBLE_DEVICES: None
Device: Tesla T4


In [43]:

ADAPTER_TYPE1_DIR = OUTPUT_DIR / "lora_type1"
model, tokenizer = train_style("type1", train_data_type1, ADAPTER_TYPE1_DIR)


Training style: type1
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 6 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
1,2.565285
2,2.425633
3,2.532436
4,2.637090
5,2.447551
6,2.138406
7,2.204741
8,2.017262
9,2.112413
10,1.749499


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_type1/checkpoints/checkpoint-150/tokenizer_config.json.


Adapter saved to: /kaggle/working/lora_type1


In [44]:
# Cell 7: Train type2 adapter

ADAPTER_TYPE2_DIR = OUTPUT_DIR / "lora_type2"
model, tokenizer = train_style("type2", train_data_type2, ADAPTER_TYPE2_DIR)


Training style: type2
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 6 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
1,2.434749
2,2.374344
3,2.382604
4,2.507575
5,2.287536
6,2.121140
7,1.904580
8,1.926054
9,1.860079
10,1.858956


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_type2/checkpoints/checkpoint-150/tokenizer_config.json.


Adapter saved to: /kaggle/working/lora_type2


In [ ]:

from peft import PeftModel
import gc, torch

def generate_for_style(style_key: str, adapter_dir: Path, test_texts: list, output_path: Path):
    global model, tokenizer

    print(f"\n{'='*60}")
    print(f"Generating style: {style_key}")
    print(f"Adapter: {adapter_dir}")
    print(f"{'='*60}")

    if 'model' in globals():
        try:
            del model
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )

    model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    FastLanguageModel.for_inference(model)

    n_lora_params = sum(p.numel() for n, p in model.named_parameters() if "lora" in n.lower())
    print(f"Loaded LoRA params: {n_lora_params:,}")

    generated = []
    for i, text in enumerate(test_texts, 1):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPTS[style_key]},
            {"role": "user", "content": text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.4,
                top_p=0.9,
                repetition_penalty=1.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        answer = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        generated.append(answer)
        print(f"  [{i}/{len(test_texts)}] OK ({len(answer)} chars)")

    with open(output_path, "w", encoding="utf-8") as f:
        for i, ans in enumerate(generated, 1):
            f.write(f"### Пример {i}\n{ans}\n\n")
    print(f"Saved: {output_path}")

    return generated


outputs_type1_path = OUTPUT_DIR / "outputs_type1.txt"
outputs_type2_path = OUTPUT_DIR / "outputs_type2.txt"

gen_type1 = generate_for_style("type1", ADAPTER_TYPE1_DIR, test_texts, outputs_type1_path)
gen_type2 = generate_for_style("type2", ADAPTER_TYPE2_DIR, test_texts, outputs_type2_path)


Generating style: type1
Adapter: /kaggle/working/lora_type1
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [2/5] OK (211 chars)


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [3/5] OK (236 chars)


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [4/5] OK (262 chars)
  [5/5] OK (297 chars)
Saved: /kaggle/working/outputs_type2.txt


In [46]:

print("="*60)
print("TYPE 1 (аналитика/финансы)")
print("="*60)
for x in gen_type1[:2]:
    print(x[:600], "\n---")

print("="*60)
print("TYPE 2 (городской новостник)")
print("="*60)
for x in gen_type2[:2]:
    print(x[:600], "\n---")

TYPE 1 (аналитика/финансы)
Россия начала тестировать цифровой рубль. Национальная валюта стала официально доступной для граждан, но пока только для банковских счетов. Тарифы за оплату услуг упали до 0,3%, что делает ее более привлекательной для бизнеса, чем электронные платежные системы. ЦБ РФ объявил, что цифровой рубль будет использоваться как дополнительный способ хранения денег, но не заменит наличные. @banki_oil 
---
Законопроект предложит вживлять микрочипы всем домашним животным. Собак и кошек будут обязывать регистрировать через три месяца после покупки. Штраф за игнорирование порядка составит до 10 тысяч рублей. Организаторы проекта уверяют, что это поможет решить проблему безнадзорных питомцев, но одновременно подвергли сомнению эффективность собачьей памяти и возможность неправильно выбрать кличку другому жителю дома. @banki_oil 
---
TYPE 2 (городской новостник)
💳 ЦБ вместе с банками запустили второй этап тестирования цифрового рубля.

Пользователи смогут открывать и пополня